# 6b: LSTM-GATv2 Rolling Pearson - Edge Perturbation Ablation

Ablation study to assess GAT robustness to edge perturbation (for comparison with 6a GCN).

**Configure at top:**
- `PERTURBATION_MODE`: "add" or "subtract"
- `PERTURBATION_FRACTION`: fraction of real edges to perturb

**Fixed:** threshold=0.5, lookback=20, equity returns, SAME base seeds as 6a for fair comparison

Run this notebook multiple times with different (mode, fraction) combinations
to produce the GAT robustness sweep. Compare to 6a results.

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral yfinance

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from empyrical import sharpe_ratio, sortino_ratio, max_drawdown, annual_return, annual_volatility, calmar_ratio
import random, tensorflow as tf
print(f"TensorFlow: {tf.__version__}")

## 2. Configuration

In [ ]:
# =============================
# PERTURBATION PARAMETERS
# =============================
PERTURBATION_MODE = "subtract"       # "add" or "subtract"
PERTURBATION_FRACTION = 0.5          # e.g. 0.25, 0.50, 0.75, 1.00
# =============================

EXPERIMENT_NAME = "6b_perturbation_gat"
SEED = 40
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023
VOL_TARGET = 0.15

# Rolling Pearson configuration (threshold=0.5 for ablation)
CORRELATION_LOOKBACK = 20
CORRELATION_THRESHOLD = 0.5
USE_EQUITY_RETURNS = True

# GATv2 hyperparameters (match 4c_GATv2_rolling_pearson.ipynb baseline exactly)
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 16
ATTN_HEADS = 4
LSTM_DROPOUT = 0.4
ATTN_DROPOUT = 0.1
LEARNING_RATE = 0.0008
MAX_GRADIENT_NORM = 1.0
BATCH_SIZE = 57
USE_RESIDUAL = True
SCALE_SCORES = True

returns_type = "equity" if USE_EQUITY_RETURNS else "straddle"
CONFIG_NAME = f"{PERTURBATION_MODE}_{PERTURBATION_FRACTION:.2f}_lb{CORRELATION_LOOKBACK}_th{CORRELATION_THRESHOLD}_{returns_type}"

print(f"Experiment: {EXPERIMENT_NAME} (seed={SEED})")
print(f"Config: {CONFIG_NAME}")
print(f"Perturbation: mode={PERTURBATION_MODE}, fraction={PERTURBATION_FRACTION}")
print(f"Residual: {USE_RESIDUAL}, Scale scores: {SCALE_SCORES}")

## 3. Helper Functions

In [ ]:
def calc_daily_returns(df, returns_col="captured_returns"):
    num_tickers = df["identifier"].nunique()
    daily_ret = df.groupby("time")[returns_col].sum() / num_tickers
    daily_ret.index = pd.to_datetime(daily_ret.index)
    return daily_ret.sort_index()

def calc_vol_scaled_returns(daily_returns, target_vol=0.15):
    v = daily_returns.std() * np.sqrt(252)
    return daily_returns * (target_vol / v) if v > 0 else daily_returns

def calc_metrics(dr, name="Strategy"):
    return {"Strategy": name, "E[Ret.]": annual_return(dr), "Vol.": annual_volatility(dr),
            "Sharpe": sharpe_ratio(dr), "Sortino": sortino_ratio(dr), "Max DD": -max_drawdown(dr),
            "Calmar": calmar_ratio(dr), "Hit Rate": (dr > 0).mean(),
            "Avg P/L": dr[dr > 0].mean() / abs(dr[dr < 0].mean()) if (dr < 0).any() else np.nan}

def calc_metrics_vol_normalized(dr, name, tv=0.15):
    s = calc_vol_scaled_returns(dr, tv)
    return calc_metrics(s, name + " (Vol-Norm)"), s

def display_metrics(m):
    df = pd.DataFrame([m]).set_index("Strategy")
    for c in ["E[Ret.]","Vol.","Max DD","Hit Rate"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.2%}")
    for c in ["Sharpe","Sortino","Calmar","Avg P/L"]:
        if c in df.columns: df[c] = df[c].apply(lambda x: f"{x:.3f}")
    display(df)

def calc_yearly_sharpes(dr):
    return {y: sharpe_ratio(dr[dr.index.year == y]) for y in sorted(dr.index.year.unique())}

## 4. Data Loading (Rolling Pearson, same as 6a)

In [ ]:
features_path = "data/straddle_features/features.csv"
if 'google.colab' in str(get_ipython()):
    features_path = "/content/drive/MyDrive/features.csv"

df = pd.read_csv(features_path)
df["date"] = pd.to_datetime(df["date"])
print(f"Loaded {len(df)} rows, {df['ticker'].nunique()} tickers")

In [ ]:
from gml.graph_model_inputs import RollingGraphModelFeatures

rolling_features = RollingGraphModelFeatures(
    df=df, total_time_steps=TOTAL_TIME_STEPS,
    correlation_lookback=CORRELATION_LOOKBACK,
    correlation_threshold=CORRELATION_THRESHOLD,
    returns_column="daily_returns",
    use_equity_returns=USE_EQUITY_RETURNS,
    start_boundary=TRAIN_START, test_boundary=TEST_START, test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True, time_features=False,
)

datasets = rolling_features.make_rolling_graph_dataset(sliding_window=True)
train_data = datasets["train"]
valid_data = datasets["valid"]
test_data  = datasets["test"]

print(f"\nBaseline shapes:")
print(f"  Train: inputs={train_data['inputs'].shape}, adj={train_data['adjacency'].shape}")
print(f"  Test:  inputs={test_data['inputs'].shape}, adj={test_data['adjacency'].shape}")

## 5. Inject Perturbation (SAME base_seeds as 6a for fair comparison)

In [ ]:
from gml.experiment_utils import perturb_adjacencies

train_adj_baseline = train_data["adjacency"].copy()
test_adj_baseline = test_data["adjacency"].copy()

# CRITICAL: use identical base_seeds as 6a notebook (42/43/44) so both models see same perturbation
print(f"Perturbing adjacencies (mode={PERTURBATION_MODE}, fraction={PERTURBATION_FRACTION})...")
train_data["adjacency"] = perturb_adjacencies(train_data["adjacency"], PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=42)
valid_data["adjacency"] = perturb_adjacencies(valid_data["adjacency"], PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=43)
test_data["adjacency"]  = perturb_adjacencies(test_data["adjacency"],  PERTURBATION_MODE, PERTURBATION_FRACTION, base_seed=44)

def count_edges(adj):
    counts = []
    for A in adj:
        A = A.copy(); np.fill_diagonal(A, 0)
        counts.append(int((A > 0).sum() / 2))
    return np.array(counts)

baseline_edges = count_edges(test_adj_baseline)
perturbed_edges = count_edges(test_data["adjacency"])

if PERTURBATION_MODE == "subtract":
    expected = (1 - PERTURBATION_FRACTION) * baseline_edges
elif PERTURBATION_MODE == "add":
    expected = (1 + PERTURBATION_FRACTION) * baseline_edges

print(f"\nTest edge counts verification:")
print(f"  Baseline: mean={baseline_edges.mean():.1f}, range=[{baseline_edges.min()}, {baseline_edges.max()}]")
print(f"  Perturbed: mean={perturbed_edges.mean():.1f}, range=[{perturbed_edges.min()}, {perturbed_edges.max()}]")
print(f"  Expected (approx): mean={expected.mean():.1f}")

## 6. Model Definition (GATv2 Rolling with Residual)

In [ ]:
from gml.graph_attention_v2 import build_lstm_gat_rolling

num_tickers = train_data["inputs"].shape[1]
time_steps  = train_data["inputs"].shape[2]
input_size  = train_data["inputs"].shape[3]

print(f"Building GATv2 rolling with perturbed adjacencies:")
print(f"  num_tickers={num_tickers}, time_steps={time_steps}, input_size={input_size}")

model = build_lstm_gat_rolling(
    num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE, gat_units=GAT_UNITS, attn_heads=ATTN_HEADS,
    lstm_dropout=LSTM_DROPOUT, attn_dropout=ATTN_DROPOUT,
    learning_rate=LEARNING_RATE, max_gradient_norm=MAX_GRADIENT_NORM,
    scale_scores=SCALE_SCORES, use_residual=USE_RESIDUAL,
)
print(f"\nTotal parameters: {model.count_params():,}")

## 7. Training

In [ ]:
X_train = [train_data["inputs"], train_data["adjacency"]]
y_train = train_data["outputs"]
w_train = train_data["active_entries"]

X_valid = [valid_data["inputs"], valid_data["adjacency"]]
y_valid = valid_data["outputs"]
w_valid = valid_data["active_entries"]

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True, verbose=1)

print("=" * 60)
print(f"Training {EXPERIMENT_NAME} ({CONFIG_NAME}, seed={SEED})")
print(f"  Perturbation: {PERTURBATION_MODE} {PERTURBATION_FRACTION}")
print("=" * 60)

history = model.fit(
    X_train, y_train, sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[early_stopping], verbose=1,
)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.xlabel("Epoch"); plt.ylabel("Loss (Neg Sharpe)")
plt.title(f"{EXPERIMENT_NAME}: {PERTURBATION_MODE} {PERTURBATION_FRACTION} Training Loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Epochs: {len(history.history['loss'])}, Best val: {min(history.history['val_loss']):.4f}")

## 8. Evaluation

In [ ]:
X_test = [test_data["inputs"], test_data["adjacency"]]
predictions = model.predict(X_test)
print(f"Predictions shape: {predictions.shape}")

positions = predictions[:, :, -1, 0].reshape(-1)
returns   = test_data["outputs"][:, :, -1, 0].reshape(-1)
captured  = positions * returns
dates     = test_data["date"][:, :, -1, 0].reshape(-1)
ids       = test_data["identifier"][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({"time": dates, "identifier": ids,
                           "position": positions, "returns": returns,
                           "captured_returns": captured})
results_df["time"] = pd.to_datetime(results_df["time"])
results_df = results_df[results_df["identifier"] != "0"]
print(f"Results: {len(results_df)} rows")

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "=" * 60)
print(f"{EXPERIMENT_NAME} ({CONFIG_NAME}) — Results")
print("=" * 60)
metrics_raw = calc_metrics(daily_returns, EXPERIMENT_NAME)
display_metrics(metrics_raw)

metrics_norm, _ = calc_metrics_vol_normalized(daily_returns, EXPERIMENT_NAME, VOL_TARGET)
display_metrics(metrics_norm)

yearly_sharpes = calc_yearly_sharpes(daily_returns)
print("\nYearly Sharpes:")
for y, s in yearly_sharpes.items():
    print(f"  {y}: {s:.4f}")

## 9. Extract Attention and Save

In [ ]:
# Extract GAT attention weights
gat_layer = model.get_layer("dynamic_masked_gat")
W_src = gat_layer.W_src.numpy()
W_dst = gat_layer.W_dst.numpy()
a = gat_layer.a.numpy()
num_heads = W_src.shape[0]

gat_input_tensor = gat_layer.input[0]
sub_model = tf.keras.Model(inputs=model.input, outputs=gat_input_tensor)

print(f"Extracting LSTM hidden states for {test_data['inputs'].shape[0]} test windows...")
lstm_outputs = sub_model.predict(X_test, verbose=0)

test_adj = test_data["adjacency"]
all_attn = []
for idx in range(lstm_outputs.shape[0]):
    h = lstm_outputs[idx, -1, :, :]
    adj = test_adj[idx]
    mask = ((adj + np.eye(adj.shape[0])) > 0).astype(np.float32)

    head_attns = []
    for head_idx in range(num_heads):
        h_s = h @ W_src[head_idx]
        h_d = h @ W_dst[head_idx]
        pw = h_s[:, np.newaxis, :] + h_d[np.newaxis, :, :]
        pw = np.where(pw > 0, pw, 0.2 * pw)
        scores = (pw @ a[head_idx]).squeeze(-1)
        scores = np.where(mask > 0, scores, -1e9)
        exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
        attn = exp_s / (exp_s.sum(axis=-1, keepdims=True) + 1e-9)
        head_attns.append(attn)
    all_attn.append(np.stack(head_attns, axis=0))

all_attn = np.array(all_attn)
print(f"Attention shape: {all_attn.shape}")

In [ ]:
from gml.experiment_utils import save_experiment_results, compute_graph_stats

test_dates_arr = pd.to_datetime(test_data["date"][:, 0, -1, 0])
all_graphs_avg = all_attn.mean(axis=1)
graph_stats = compute_graph_stats(all_graphs_avg, threshold=0.0)

hyperparams = {
    "model_type": "GATv2_rolling_pearson_perturbed",
    "hidden_layer_size": HIDDEN_LAYER_SIZE, "gat_units": GAT_UNITS,
    "attn_heads": ATTN_HEADS, "lstm_dropout": LSTM_DROPOUT,
    "attn_dropout": ATTN_DROPOUT, "learning_rate": LEARNING_RATE,
    "max_gradient_norm": MAX_GRADIENT_NORM, "batch_size": BATCH_SIZE,
    "use_residual": USE_RESIDUAL, "scale_scores": SCALE_SCORES,
    "correlation_lookback": CORRELATION_LOOKBACK,
    "correlation_threshold": CORRELATION_THRESHOLD,
    "use_equity_returns": USE_EQUITY_RETURNS,
    "perturbation_mode": PERTURBATION_MODE,
    "perturbation_fraction": PERTURBATION_FRACTION,
    "total_time_steps": TOTAL_TIME_STEPS,
    "train_start": TRAIN_START, "test_start": TEST_START, "test_end": TEST_END,
}

save_experiment_results(
    experiment_name=EXPERIMENT_NAME, seed=SEED, config_name=CONFIG_NAME,
    predictions=predictions, results_df=results_df,
    daily_returns=daily_returns, metrics_raw=metrics_raw,
    metrics_norm=metrics_norm, yearly_sharpes=yearly_sharpes,
    training_history=history.history, hyperparams=hyperparams,
    test_dates=test_dates_arr.values,
    attention_weights=all_attn,
    adjacency=test_data["adjacency"],
    graph_stats=graph_stats,
    model=model,
    base_dir=RESULTS_BASE,
)

## 10. Summary

In [ ]:
print("=" * 60)
print(f"EXPERIMENT: {EXPERIMENT_NAME}")
print(f"Config: {CONFIG_NAME}")
print("=" * 60)
print(f"Perturbation: {PERTURBATION_MODE} {PERTURBATION_FRACTION}")
print(f"Baseline edges (test mean): {baseline_edges.mean():.1f}")
print(f"Perturbed edges (test mean): {perturbed_edges.mean():.1f}")
print(f"\nSharpe: {metrics_raw['Sharpe']:.3f}")
print(f"Return: {metrics_raw['E[Ret.]']:.2%}, Vol: {metrics_raw['Vol.']:.2%}")
print(f"Saved to: {RESULTS_BASE}/{EXPERIMENT_NAME}/{CONFIG_NAME}/seed_{SEED}/")